# Session 1: from a catalogue to a defensible result

Our 400-row galaxy catalogue is **synthetic teaching data**: input distances $d$ are exact and sampled uniformly across $20\text{–}200\text{ Mpc}$ (a flat toy baseline, not a homogeneous-volume $dN/dd \propto d^2$ survey), and recession velocities follow $v = (70\text{ km s}^{-1}\text{ Mpc}^{-1})\,d + 300\text{ km s}^{-1}$ with independent Gaussian measurement noise $\sigma_v$. It is not a cosmological measurement.

By the end of this session you will be able to:
- verify that your Jupyter kernel and terminal share the same isolated, solver-locked Python environment;
- filter and broadcast catalogue arrays with one shared boolean mask so rows and physical columns stay aligned;
- check Astropy units explicitly and preserve provenance cards across an ECSV-to-FITS table round-trip;
- save a publication-ready multi-panel figure and caption distinguishing data-quality hygiene from survey selection.

## Environments and kernels: pip, conda and uv

- **Why isolate & solve dependencies:** Scientific libraries form a constraint graph (for example, `astropy` and `scipy` requiring compatible `numpy` bounds). A **dependency solver (resolver)** searches the full graph for one mutually consistent set of package versions before installing:
  - Ad-hoc `pip install pkg` resolves only against whatever is currently installed (with no lockfile by default), so adding a package months later can silently upgrade `numpy` and break working research code.
  - **`uv`** *(used here)* resolves the full `pyproject.toml` graph in milliseconds and pins exact versions plus cryptographic hashes in `uv.lock` (`uv sync --frozen --extra teaching`).
  - **`conda` / `mamba`** (`libmamba` SAT solver) resolves Python packages **jointly with compiled C/Fortran/CUDA libraries** (`CFITSIO`, `FFTW`, MPI/CUDA stacks on `conda-forge`).

![Diagram: Python environments, package managers and Jupyter kernels](../book/diagrams/python-environments.svg)

**One environment per project, registered as a named kernel** ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/python-environments.dot)).

- **`Kernel != terminal` (and hidden notebook state):**
  - A notebook kernel runs one `sys.executable` (`/opt/course/.venv/bin/python` on CPD), whereas opening a terminal shell does **not** activate that environment automatically — run `source .../activate` or `uv run` first.
  - Never `pip install` into system Python or Conda `(base)`. In a `uv` project use `uv add` / `uv sync` (`uv` omits `pip` from `.venv` by default; in standard `venv`s use `python -m pip`, or `%pip` inside notebooks rather than `!pip`). In mixed Conda environments, install `conda-forge` packages *before* `pip` because `pip`'s solver cannot re-solve compiled Conda libraries.
  - Notebooks keep **hidden variable state** when cells execute out of order — always verify finished work via **Restart Kernel and Run All Cells**.

:::{dropdown} Reference table: pip + venv vs uv vs conda / mamba
| Tool | What its solver manages | Spec & lock files | Best research fit |
| --- | --- | --- | --- |
| **`pip` + `venv`** | Python wheels/sdists against current env (no default lockfile) | `pyproject.toml`, `requirements.txt` | Simple pure-Python / wheel-only setups |
| **`uv`** *(used here)* | Python interpreter + `.venv` + fast full-graph PyPI resolver | `pyproject.toml` + version/hash-pinned `uv.lock` | Fast, reproducible project environments |
| **`conda` / `mamba`** | Joint SAT solve across Python + compiled C/Fortran/CUDA libs | `environment.yml` (`conda-lock.yml`) | Binary HPC/astro stacks (`CFITSIO`, `FFTW`, MPI/CUDA) |
:::

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable

print('Interpreter:', sys.executable, '| NumPy:', np.__version__)
plt.rcParams.update({
    'figure.figsize': (7, 4), 'font.size': 11,
    'xtick.direction': 'in', 'ytick.direction': 'in', 'xtick.top': True, 'ytick.right': True,
})

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path('/opt/course'), Path.home() / 'jupyter_data_languages']
     if (p / 'data/mock_galaxies.ecsv').is_file()),
    None,
)
if ROOT is None:
    import course2026
    ROOT = Path(course2026.__file__).resolve().parents[1]

## Inspect the catalogue

We use Astropy's `QTable` because it keeps `Quantity` column units and FITS/ECSV header metadata attached to the data (with `.to_pandas()` available whenever flat export is needed). Our `mock_galaxies.ecsv` **Extended CSV** file pairs plain-text rows with a `#`-commented YAML header; printed columns show full IEEE-754 binary-float precision until explicitly formatted.

Before running the next cell, predict what columns, units and provenance keys (`table.meta`) a survey or simulation catalogue should record.

In [ ]:
table = QTable.read(ROOT / 'data/mock_galaxies.ecsv')
print('Rows:', len(table), '| metadata:', table.meta)
table[:3].pprint_all()

## Arrays, shapes and masks

- **One shared boolean mask:** never filter columns separately — build a single mask `valid`, inspect `~valid` (boolean NOT), and slice `table[valid]` or every extracted array together so galaxies stay aligned.
- **Unit-checked extraction:** `.to_value(u.Mpc)` verifies dimensional compatibility and converts units before returning a plain `float` NumPy array, whereas `.value` strips units blindly.
- **Real-survey sentinels:** `np.isfinite(...)` catches `NaN` and `Inf`, but archive catalogues also encode missing entries with sentinel numbers (`-99`, `-999`, `99.0`) or `MaskedColumn` flags — always check column docs and physical bounds (`distance > 0`, `sigma > 0`).

![Diagram: One mask preserves row alignment](../book/diagrams/shared-mask.svg)

**Keep objects aligned:** separate per-column filters can silently pair one galaxy's distance with another's velocity ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/shared-mask.dot)).

In [ ]:
distance = table['distance'].to_value(u.Mpc)
velocity = table['velocity'].to_value(u.km / u.s)
sigma = table['velocity_error'].to_value(u.km / u.s)
magnitude = np.asarray(table['apparent_magnitude'])

# Joint validity mask for velocity + magnitude-selection analysis (excludes 1 row with NaN magnitude)
valid = (
    np.isfinite(distance) & np.isfinite(velocity) & np.isfinite(sigma)
    & np.isfinite(magnitude) & (distance > 0) & (sigma > 0)
)
clean = table[valid]  # slices all QTable columns and Quantity units together
print('Rows:', len(table), '| usable:', len(clean), '| removed:', np.count_nonzero(~valid))
assert len(clean) == 395
assert distance[valid].shape == velocity[valid].shape == sigma[valid].shape

### Task A

1. Using `np.count_nonzero(...)`, count missing (`~np.isfinite`) velocities and invalid (`~np.isfinite(sigma) | (sigma <= 0)`) velocity errors separately, and inspect all 5 excluded rows with `table[~valid]`.
2. Explain why replacing missing velocities with `0.0` (or leaving `-999` sentinel values unmasked) would corrupt a mean or linear fit.

In [ ]:
# Task A: count missing velocities and invalid velocity errors separately, then inspect table[~valid]

### Shapes and broadcasting

- **Right-to-left alignment:** NumPy compares shapes from the trailing axis leftward; `(M, 1)` and `(N,)` align as `(M, 1) * (1, N) -> (M, N)`, evaluating every trial slope across every galaxy without Python loops.
- **Shape-mismatch & square-matrix trap:** `h0_grid * distance[valid]` (`(201,) * (395,)`) raises `ValueError` because `201 != 395` — *take care when `M == N` (e.g. `(3, 3) * (3,)`), which silently scales columns instead of rows!*
- **Axis reductions:** `.sum(axis=1)` collapses across the `395` galaxy columns (`axis=1`) to return one $\chi^2$ value per trial slope `(201,)` (see the [Python Primer](00-primer.ipynb) for 2D `axis=0` vs `axis=1`).

![Diagram: Broadcast trial slopes across galaxies](../book/diagrams/broadcasting.svg)

**Outer-grid evaluation:** reshaping `(M,)` to `(M, 1)` evaluates $M$ trial slopes across $N$ galaxies in one `(M, N)` array ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/broadcasting.dot)).

In [ ]:
h0_grid = np.linspace(60.0, 80.0, 201)  # (201,) trial slopes [km/s/Mpc]
v_model = h0_grid[:, None] * distance[valid] + 300.0  # (201, 1) * (395,) -> (201, 395)
chi2 = (((velocity[valid] - v_model) / sigma[valid]) ** 2).sum(axis=1)  # collapse galaxies -> (201,)
best_h0 = h0_grid[chi2.argmin()]
print(f'v_model: {v_model.shape} | chi2: {chi2.shape} | grid best H0 = {best_h0:.1f} km/s/Mpc')
assert v_model.shape == (201, 395) and chi2.shape == (201,) and 68.0 < best_h0 < 72.0

## Units and FITS provenance

Never guess units from numeric magnitudes: attach Astropy `Unit` objects so dimensional algebra (such as converting inverse Hubble slope $1 / H_0$ from $\text{Mpc s km}^{-1}$ to $\text{Gyr}$, or `km/s` to `m/s`) is checked automatically and physically incompatible conversions raise `UnitConversionError`.

In [ ]:
H0 = 70.0 * u.km / (u.s * u.Mpc)
print(f'Hubble time 1/H0 = {(1 / H0).to(u.Gyr):.2f}')

velocity_ms = clean['velocity'].to(u.m / u.s)
assert u.allclose(velocity_ms, (1000.0 * velocity[valid]) * (u.m / u.s))
try:
    clean['distance'].to(u.s)
except u.UnitConversionError:
    print('Expected: a distance cannot be converted to seconds')

Survey archives store binary tables and images in **FITS (Flexible Image Transport System)** as numbered Header/Data Units: an empty Primary Header (`HDU 0`) followed by a Binary Table (`HDU 1`) whose `TUNITn` cards (such as `'km s-1'`) preserve column units. Standard FITS headers expect `<=8`-character keywords (longer keys trigger `HIERARCH` warnings and nested `meta` dicts fail), so rather than wiping provenance, map essential metadata into short FITS header cards.

![Diagram: ECSV text metadata vs FITS Header and Data Units](../book/diagrams/fits-vs-ecsv.svg)

**Text ECSV vs binary FITS:** both preserve `TUNITn` / YAML column units; mapping short FITS keywords retains provenance cleanly ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/fits-vs-ecsv.dot)).

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / 'mock_subset.fits'
    subset = clean[['object_id', 'distance', 'velocity', 'velocity_error']].copy()
    subset.meta = {
        'SYNTH': bool(table.meta['synthetic']),
        'SEED': int(table.meta['seed']),
        'H0_INJ': float(table.meta['slope']),
    }
    subset.write(path)
    restored = QTable.read(path)
    np.testing.assert_allclose(restored['distance'].to_value(u.Mpc), distance[valid])
    assert restored.meta['SEED'] == 20261109
    print('FITS rows:', len(restored), '| unit:', restored['distance'].unit, '| meta:', restored.meta)

### Task B

1. Compare `table` and `restored`: which catalogue column was omitted, and how did `SEED` and `H0_INJ` survive in `restored.meta`?
2. Explain to a partner why `restored['distance'].to_value(u.Mpc)` is safer than `.value` if an external FITS table stored distances in `kpc`, then write two `assert` checks for reading an external FITS catalogue.

*If skipping the two optional spotlights below, take the 10-minute **Break** after Task B.*

In [ ]:
# Task B: write two assertions you would run immediately after reading an external FITS catalogue

### Optional spotlight: physical constants and spectral equivalencies

*Optional reference spotlight — skip ahead to **Break** if live workshop time is tight.*

`astropy.constants` supplies CODATA/IAU constants with full units (preventing CGS-vs-SI factor errors in theoretical derivations such as $R_s = 2GM_\odot/c^2$), while `equivalencies` translate observed spectral wavelengths into frequencies (`u.spectral()`) or optical recession velocities $v_{\text{opt}} = c\,\Delta\lambda / \lambda_0$ (`u.doppler_optical`).

In [ ]:
from astropy import constants as const

r_s = (2 * const.G * const.M_sun / const.c**2).to(u.km)
obs_ha, rest_ha = 660.0 * u.nm, 656.28 * u.nm
freq_ha = obs_ha.to(u.THz, equivalencies=u.spectral())
v_opt = obs_ha.to(u.km / u.s, equivalencies=u.doppler_optical(rest_ha))
print(f'Solar R_s = {r_s:.2f} | H-alpha {obs_ha} -> {freq_ha:.1f} -> v_opt = {v_opt:.1f}')

### Optional spotlight: 2D FITS cutouts, WCS and colormaps

*Optional reference spotlight — skip ahead to **Break** if live workshop time is tight.*

FITS image HDUs pair 2D sky cutouts or simulation slices with World Coordinate System (`WCS`) header keywords:
- Pass `origin='lower'` so pixel `(0, 0)` sits at bottom-left (NumPy indexes `[row_y, col_x]` from `0`, whereas FITS `CRPIX` is `(x, y)` from `1`, and sky `RA` increases eastward to the left).
- Stretch high-dynamic-range intensity with `ImageNormalize(..., PercentileInterval(99.5), stretch=AsinhStretch())` so bright cores do not hide faint companions.
- Pair **perceptually uniform** colormaps (`magma`, `viridis`) with positive intensity, **zero-centred diverging** colormaps (`RdBu_r` + `CenteredNorm(vcenter=0)`) with signed residuals, and avoid `jet` *(micro-check: try `origin='upper'` to watch pixel `(0, 0)` flip)*.

**Break** *(10 minutes — resume at "Plot the data and its uncertainty").*

In [ ]:
from matplotlib.colors import CenteredNorm
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize, PercentileInterval
from astropy.wcs import WCS

rng = np.random.default_rng(7)  # fixed seed for reproducible synthetic sky noise
yy, xx = np.mgrid[:64, :64]
smooth_model = 480 * np.exp(-0.5 * (((xx - 31.5) / 3.2) ** 2 + ((yy - 31.5) / 2.4) ** 2)) + 36 * np.exp(-0.5 * (((xx - 31.5) / 13) ** 2 + ((yy - 31.5) / 9.5) ** 2))
cutout = smooth_model + 32 * np.exp(-0.5 * (((xx - 45) / 2.2) ** 2 + ((yy - 22) / 2.2) ** 2)) + rng.normal(0.0, 2.5, (64, 64))

wcs = WCS(naxis=2)
wcs.wcs.crpix, wcs.wcs.cdelt = [32.5, 32.5], [-0.00028, 0.00028]
wcs.wcs.crval, wcs.wcs.ctype = [150.119, 2.205], ['RA---TAN', 'DEC--TAN']
hdu = fits.PrimaryHDU(data=cutout, header=wcs.to_header())

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8), layout='constrained', subplot_kw={'projection': WCS(hdu.header)})
norm_img = ImageNormalize(hdu.data, interval=PercentileInterval(99.5), stretch=AsinhStretch())
im0 = axes[0].imshow(hdu.data, origin='lower', cmap='magma', norm=norm_img)
fig.colorbar(im0, ax=axes[0], ticks=[0, 50, 150, 350], label='Flux [arb.]')
axes[0].set(xlabel='RA', ylabel='Dec', title='FITS cutout (asinh stretch)')

im1 = axes[1].imshow(hdu.data - smooth_model, origin='lower', cmap='RdBu_r', norm=CenteredNorm(vcenter=0.0))
fig.colorbar(im1, ax=axes[1], label='Residual [arb.]')
axes[1].set(xlabel='RA', ylabel='Dec', title='Data − smooth model (zero-centred)')
for ax in axes:
    ax.coords[0].set_ticks(spacing=0.006 * u.deg)
    ax.coords[0].set_major_formatter('d.ddd')
    ax.coords[1].set_major_formatter('d.ddd')
plt.show()

## Plot the data and its uncertainty

- **Explicit `Figure` + `Axes` objects:** use `fig, axes = plt.subplots(...)` rather than stateful `plt.*` calls so every curve, limit and legend targets a named panel.
- **Full sample + legible error bars:** plot all 395 valid galaxies faintly in the background alongside `[::8]` subsampled $1\sigma_v$ error bars (`n=50`; $d$ has no horizontal error bars because input distances are exact); the `fill_between` band shows **illustrative $\pm\text{median }\sigma_v$ measurement scatter**, not parameter-fit uncertainty.
- **Inverted astronomical magnitudes:** because $m = -2.5\log_{10} F + \text{const}$, larger magnitudes mean fainter flux ($F \propto L / 4\pi d^2$) — call `ax.invert_yaxis()` so brighter galaxies sit higher.
- **Persistent publication export:** write vector `.pdf` for LaTeX papers and `dpi=300` `.png` for slides into `outputs/` (use `rasterized=True` or `ax.hexbin` when plotting $\ge 10^5$ survey stars or MCMC samples).

In [ ]:
magnitude_limit = 16.5
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
d_grid = np.linspace(20, 200, 200)
v_line = 70.0 * d_grid + 300.0
med_sigma = np.median(sigma[valid])
indices = np.flatnonzero(valid)[::8]

axes[0].fill_between(
    d_grid, v_line - med_sigma, v_line + med_sigma,
    color='C0', alpha=0.18, label=rf'Injected line $\pm$ median $\sigma_v$ ({med_sigma:.0f} km/s)',
)
axes[0].plot(d_grid, v_line, color='C0', lw=1.6)
axes[0].scatter(distance[valid], velocity[valid], s=10, color='0.75', alpha=0.35, label=f'All valid (n={valid.sum()})')
axes[0].errorbar(
    distance[indices], velocity[indices], yerr=sigma[indices],
    fmt='o', color='0.15', ms=3.5, elinewidth=1.0, capsize=2,
    label=rf'Every 8th with $1\sigma_v$ (n={len(indices)})',
)
axes[0].set(xlabel='Distance (exact input) [Mpc]', ylabel='Velocity [km/s]', title=r'Velocity relation and $1\sigma_v$ errors')
axes[0].legend(fontsize=8.2, loc='upper left')

bright = valid & (magnitude < magnitude_limit)
faint = valid & (magnitude >= magnitude_limit)
axes[1].scatter(distance[faint], magnitude[faint], s=14, color='0.75', alpha=0.8, label=f'Valid, not selected m ≥ {magnitude_limit} (n={faint.sum()})')
axes[1].scatter(distance[bright], magnitude[bright], s=14, color='C0', alpha=0.8, label=f'Selected m < {magnitude_limit} (n={bright.sum()})')
axes[1].axhline(magnitude_limit, color='C3', ls='--', lw=1.4, label=f'Flux limit m = {magnitude_limit}')
axes[1].invert_yaxis()
axes[1].set(xlabel='Distance (exact input) [Mpc]', ylabel='Apparent magnitude [mag]', title='Magnitude cut removes distant galaxies')
axes[1].legend(fontsize=8.2, loc='lower left')
fig.tight_layout()

out_dir = Path.cwd() / 'outputs'
out_dir.mkdir(exist_ok=True)
pdf_path, png_path = out_dir / 'catalogue_summary.pdf', out_dir / 'catalogue_summary.png'
fig.savefig(pdf_path)
fig.savefig(png_path, dpi=300)
assert pdf_path.stat().st_size > 0 and png_path.stat().st_size > 0
print('Saved figures:', pdf_path, 'and', png_path.name)
plt.show()

### Your turn

Plot the `[::8]` subsampled velocity residuals `resid` against `distance[valid]` with `yerr=sigma[valid][::8]` and a horizontal `ax.axhline(0.0, ls='--')` reference line, then check whether the residuals show any systematic trend with distance *(stretch: colour all 395 points by `magnitude[valid]` with `fig.colorbar`, or check what fraction of `resid / sigma[valid]` lies within `[-1, 1]`).*

In [ ]:
resid = velocity[valid] - (70.0 * distance[valid] + 300.0)
# Your turn: plot subsampled error bars around ax.axhline(0.0, ls='--')

## Investigate a survey selection

Keep data-quality hygiene (`valid`, 395 rows) distinct from survey selection (`selected`, `m < 16.5`, 262 rows) on **identical histogram bin edges**. Raw counts show where galaxies drop out; `density=True` normalises each histogram's **total bin area to 1**, so a narrower selected sample rises higher at small distances despite containing fewer objects.

Observe how cutting on `m < 16.5` shifts the mean distance $\bar{d}$, and consider before Task C whether truncating $p(d)$ biases the conditional velocity relation $E[v \mid d]$ when $d$ is exact and velocity noise is independent of $m$ (unlike classical Malmquist bias, where distances are inferred from noisy standard candles).

![Diagram: Filtering and survey selection are separate decisions](../book/diagrams/catalogue-selection.svg)

**Data hygiene vs survey cut:** changing `magnitude_limit` alters `selected` without changing which 395 rows are valid ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/catalogue-selection.dot)).

In [ ]:
selected = valid & (magnitude < magnitude_limit)
print('Selected:', selected.sum(), 'of', valid.sum(), 'valid rows')
assert selected.sum() == 262

bins = np.linspace(20, 200, 13)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, density, title in zip(axes, [False, True], ['Counts per bin', 'Normalised probability density']):
    ax.hist(distance[valid], bins=bins, histtype='step', lw=2, density=density, label=f'All valid (n={valid.sum()})')
    ax.hist(distance[selected], bins=bins, histtype='step', lw=2, density=density, label=f'm < {magnitude_limit} (n={selected.sum()})')
    ax.set(title=title, xlabel='Distance [Mpc]', ylabel='Probability density [1/Mpc]' if density else 'Objects per bin')
    if density:
        ax.set_ylim(0, 0.0108)
    ax.legend(fontsize=8.5)
fig.tight_layout()
plt.show()
print('Mean distances [Mpc] (all valid vs selected):', distance[valid].mean().round(1), distance[selected].mean().round(1))

### Task C

1. Predict how tightening the cut to `15.5` or relaxing it to `17.5` changes both the selected galaxy count and mean distance $\bar{d}$, then complete the loop below to verify numerically (leaving `selected` unchanged).
2. Draft a three-sentence figure caption stating (i) what populations and binning are plotted, (ii) what invalid rows were excluded, and (iii) why shifting $p(d)$ does **not** bias $v(d)$ under this mock's exact-distance noise model — then name one realistic survey effect that *would* bias the fit.

In [ ]:
for test_limit in (15.5, 16.5, 17.5):
    mask_cut = valid & (magnitude < test_limit)
    # Print test_limit, selected count, and mean distance [Mpc]

## Exit check

Confirm with a partner that you can explain:
- why `uv.lock` / `conda` dependency solvers prevent drift and why a notebook's `sys.executable` differs from an unactivated shell;
- why one shared `valid` mask leaves 395 usable rows and how `(201, 1) * (395,)` broadcasts across a parameter grid;
- why `.to_value(u.Mpc)` and short FITS header cards (`SEED`, `H0_INJ`) protect physical provenance better than `.value` or cleared metadata;
- why cutting at `m < 16.5` truncates $p(d)$ without biasing $E[v \mid d]$ here, and one claim this synthetic figure cannot support.

Run **Restart Kernel and Run All Cells** to verify clean top-to-bottom execution, then compare with [solutions](../book/solutions.md).